# ContractRisk — segundo juez: regresión Mistral

Sesión nueva de Colab con GPU T4. Evalúa las mismas veinte anclas aprobadas y seis pares de verbosidad: 38 juicios. Solo cambia el checkpoint del juez a Mistral-7B-Instruct-v0.3, familia distinta de Qwen. Rúbrica, criterios, cuantización y generación se mantienen.

La calibración Qwen anterior falló y se conserva intacta. No se usa para aprobar calidad. Esta prueba comprueba si el segundo modelo distingue los casos; no se garantiza mejora.

Ejecutar en orden y habilitar acceso del notebook a WANDB_API_KEY. No pegar claves ni cambiar versiones/modelos por fallos. Un kernel CUDA NF4 se comprueba antes de descargar/cargar el checkpoint.

Aunque supere esta regresión, judge_validated_for_final permanece false: faltarán confirmación separada con ejemplos nuevos revisados y auditoría de razones. Los doce ejemplos nuevos NO se ejecutan aquí. No modifica main ni Space.


In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_judge02_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-calibration.txt'], 'installation.log')
import torch
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('20 anclas originales aprobadas; hashes y rúbrica comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
RESULTS = ROOT / 'judge02_regression_results'
run_visible([sys.executable, '-u', 'final/run_second_judge.py', '--output', str(RESULTS), '--require-wandb'], 'judge02.log')


## Inspección y conservación

completed_regression_failed significa que no pasó, aunque el programa haya terminado. No repetir buscando notas favorables. completed_regression_passed_confirmation_pending no autoriza todavía al juez final: falta confirmación separada y revisión de razones.

Conservar también los fallos desde las últimas celdas. No ajustar las puntuaciones del modelo.


In [ ]:
RESULTS = ROOT / 'judge02_regression_results'
for name in ['metadata.json', 'regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
shutil.copy2(ROOT / 'final/judge_second_config.json', RESULTS / 'judge_second_config.json')
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez02_resultados'), 'zip', RESULTS)
files.download(package)
